# BioLangFusion Phase A: matched benchmark comparisons

**Question.** On a supplied benchmark split, how do BioLangFusion's fusion methods compare with their constituent single encoders across training seeds?

Step 3 builds and checks the models. This Stage 4 notebook inspects saved fungal-expression or E. coli classification results for the appendix-convolution comparison declared in Setup. It checks the recorded experiment, inventories local evidence, selects models using validation scores, and prepares tables and figures only when that comparison and its input-header checks are complete. Training remains disabled. The [retained runner](run_blf_phaseA.py) identifies the producing workflow; a local inventory is not a queue of new experiments.

Each row supplies a sequence and an expression target or class. DNA, RNA, and protein inputs are derived views of that sequence, not three independently measured assays. A fusion gain would show predictive benefit under this evaluation; it would not by itself establish biological alignment or a causal mechanism.

This notebook covers two benchmarks, not the full five-benchmark reproduction. The research plan places reproduction before interpreting gains and introducing more complex designs: [Gantt workbook](../../docs/Research_Project_Plan_Gantt.xlsx), `Details` items 17 and 19.

---

## Setup

Run from this notebook's folder or another folder inside the repository, using the project's notebook environment. The path block locates `Code/shared_code/`; it does not depend on a personal drive path.

The imports reuse the maintained implementation:

| Source | Role here |
| --- | --- |
| [`datasets.py`](../shared_code/datasets.py) | Read the same benchmark rows and supplied splits as Step 3. |
| [`train.py`](../shared_code/fusion/train.py) | Use `TrainConfig`, `Result.load`, and `paired_bootstrap` for saved-result comparisons; never call the training loop. |
| [`blf.py`](../shared_code/fusion/blf.py) | Read the modality-to-encoder mapping. |
| [`token_embeddings.py`](../shared_code/fusion/token_embeddings.py) | Reuse input and file fingerprints, without opening embedding arrays. |
| [`run_blf_phaseA.py`](run_blf_phaseA.py) | Reuse result filenames and stored-prediction validation; importing it does not run its execution entry point. |

Importing these modules requires PyTorch and SciPy, but does not load pretrained weights or start training. NumPy, pandas, and Matplotlib support the comparison tables and figures. The setup code separates editable inspection choices, expected benchmark properties, and the recorded comparison plan. The qualified `phasea_runner` helper calls make their source visible; importing that module does not invoke its command-line entry point.

In [ ]:
import json
import sys
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

# Make shared_code importable from stage4/ as well as the repository root.
working_directory = Path.cwd().resolve()
REPO_ROOT = next(
    folder for folder in (working_directory, *working_directory.parents)
    if (folder / "Code" / "shared_code" / "datasets.py").is_file()
)
CODE_DIR = REPO_ROOT / "Code"
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

from stage4 import run_blf_phaseA as phasea_runner
from shared_code import datasets
from shared_code.encoders import ENCODERS
from shared_code.fusion import blf, token_embeddings, train
from shared_code.fusion.token_embeddings import (
    file_fingerprint, input_fingerprint, inputs_for_rows,
)

# A kernel can retain imports from another checkout even after sys.path changes.
for module in (datasets, blf, token_embeddings, train):
    if not Path(module.__file__).resolve().is_relative_to(CODE_DIR / "shared_code"):
        raise RuntimeError("Shared modules came from another checkout; restart this notebook's kernel.")
if not Path(phasea_runner.__file__).resolve().is_relative_to(CODE_DIR / "stage4"):
    raise RuntimeError("The Phase A helper came from another checkout; restart this notebook's kernel.")

In [ ]:
BENCHMARK = "fungal_expression"  # Or "ecoli_proteins"; review one benchmark at a time.
SEEDS = (0, 1, 2)
POSITIVE_LAMBDAS = (0.01, 0.5, 1.0)
RUN_TRAINING = False
BOOTSTRAP_DRAWS = 2000
BOOTSTRAP_SEED = 0

# This notebook is a reader: changing the flag cannot accidentally launch fitting.
if RUN_TRAINING:
    raise RuntimeError("This notebook only inspects saved results; training remains disabled.")

STAGE3_DIR = CODE_DIR / "stage3"
# In a separate worktree, set this to the existing Stage 3 embeddings folder; do not copy caches.
# Missing headers permit inspection, but block the final comparison.
CACHE_ROOT = STAGE3_DIR / "embeddings"
RESULT_DIRECTORY = (
    STAGE3_DIR / "results" / BENCHMARK / "reproducible" if BENCHMARK == "fungal_expression"
    else CODE_DIR / "stage4" / "results" / "blf_phaseA" / BENCHMARK
)

In [ ]:
# Expected task, metric, dimensions, and split counts for the selected benchmark.
BENCHMARK_SETTINGS = {
    "fungal_expression": {
        "task": "regression", "n_outputs": 1, "label": "Fungal expression",
        "splits": {"train": 2153, "val": 555, "test": 430},
        "metric": "Spearman correlation", "display_scale": 1.0,
        "score_label": "Test Spearman correlation", "difference_label": "Difference in test Spearman",
    },
    "ecoli_proteins": {
        "task": "classification", "n_outputs": 3, "label": "E. coli expression classes",
        "splits": {"train": 3010, "val": 728, "test": 712},
        "metric": "accuracy", "display_scale": 100.0,
        "score_label": "Test accuracy (%)", "difference_label": "Accuracy difference (percentage points)",
    },
}
benchmark_settings = BENCHMARK_SETTINGS[BENCHMARK]
TASK = benchmark_settings["task"]
EXPECTED_OUTPUTS = benchmark_settings["n_outputs"]
EXPECTED_SPLITS = benchmark_settings["splits"]
METRIC = benchmark_settings["metric"]
DISPLAY_SCALE = benchmark_settings["display_scale"]
SCORE_MINIMUM = 0.0 if TASK == "classification" else -1.0
COHORT_FIELDS = (
    "version", "model_kwargs", "widths", "n_outputs", "inputs", "environment", "code_sha256",
)
EXPECTED_WIDTHS = {"dna": 512, "rna": 640, "prot": 320}

In [ ]:
# These settings identify the recorded appendix comparison, not a new training request.
BASE_CONFIG = train.TrainConfig(
    lr=3e-5, weight_decay=1e-5, batch_size=32, max_epochs=500,
    early_stopping_patience=20, plateau_patience=5,
    channels=100, tconv="appendix", entropy_lambda=0.0, seed=0,
)

# Select a historical reference explicitly; an absent file does not authorize a new fit.
REFERENCE_RESULT_PATH = phasea_runner.result_path(RESULT_DIRECTORY, BENCHMARK, "concat", BASE_CONFIG)

# Only these native-resolution models can be the comparator in this recorded analysis.
BASELINE_VARIANTS = ("unimodal:dna", "unimodal:rna", "unimodal:prot")

# Declared order also breaks exact validation ties: DNA, RNA, protein; then lower lambda.
CANDIDATES = [
    ("DNA", "unimodal:dna", 0.0),
    ("RNA", "unimodal:rna", 0.0),
    ("Protein", "unimodal:prot", 0.0),
    ("Concat", "concat", 0.0),
    *[(f"MIL lambda={weight:g}", "mil", weight) for weight in POSITIVE_LAMBDAS],
    ("MIL without entropy", "mil", 0.0),
    ("Cross", "cross", 0.0),
]
plan = pd.DataFrame(
    [(seed, label, variant, weight) for seed in SEEDS for label, variant, weight in CANDIDATES],
    columns=["seed", "candidate", "variant", "entropy_lambda"],
)
KEY_COLUMNS = ["seed", "variant", "entropy_lambda"]
planned_keys = set(plan[KEY_COLUMNS].itertuples(index=False, name=None))
print(f"Historical comparison: {benchmark_settings['label']}, {BASE_CONFIG.tconv} convolution, "
      f"{len(plan)} expected records across {len(SEEDS)} seeds. Training disabled.")
print("Metric:", METRIC, "| results:", RESULT_DIRECTORY, "| optional headers:", CACHE_ROOT)

### What stays fixed, and what is selected?

All candidates use the same retained rows, supplied train/validation/test assignments, and frozen NT-v2 100M, RNA-FM, and ESM-2 8M embeddings. The fusion layers and prediction heads are trainable in the producing runs. For each seed, the candidate fits are the three single encoders, concatenation, the declared positive MIL entropy weights, MIL without entropy, and cross-attention.

The settings above follow the project's implementation of [BioLangFusion](../../Papers/core/BioLangFusion.pdf), Appendix A.4, List 3: Adam, MSE and Spearman correlation for fungal regression, or cross-entropy and accuracy for E. coli classification. The shared loop uses the validation score for scheduling, early stopping, and selecting the epoch whose test score is saved. The paper does not fully specify the monitored quantity. A seed labels a training realization; the same seed across architectures does not imply identical initial weights or dropout masks. The training seed identifies the producing fit. `BOOTSTRAP_SEED` separately controls reproducible resampling of saved test rows; it never changes a trained model.

**Recorded comparison.** The configuration here identifies the appendix transposed convolution and 100 channels. Equation 1 and the appendix differ on convolution settings, and the MIL listing uses a different channel width. Matching array lengths does not make these alternatives equivalent; test scores must not choose the reproduction specification. The native-resolution baselines listed in `BASELINE_VARIANTS` are the eligible comparators for this analysis. The paper does not fully specify the baseline treatment, so these choices remain reproduction qualifications.

**Intended reproduction.** The team protocol specifies Equation 1 and an additional RNA-FM control pooled to codons: average each consecutive triplet of nucleotide embeddings, then apply the existing TextCNN. Comparing this control with native RNA helps distinguish the effect of RNA resolution from the benefit of combining encoders. Its implementation and comparator eligibility must be integrated explicitly; adding a model must not silently redefine "best single encoder." Changing this reader's configuration would neither convert appendix predictions into Equation 1 results nor complete that control.

This recorded analysis repeats the entire candidate grid and validation selection within each seed. A protocol that repeats only selected models on later seeds, or reports a seed-averaged paired interval, requires its own declared selection and uncertainty rules. The per-seed analysis below remains unchanged; [run records](../../docs/run-records.md) retain the producing experiment and the integration decisions.

---

## Check the inputs behind the results

The shared loader strips and uppercases sequences, keeps rows at most 999 nucleotides long, and preserves the supplied split assignments. This check reads the CSV, not the embeddings. Counts alone cannot identify a dataset: the fingerprint below includes every retained sequence, length, target, and split in order, using the same serialization as `BenchmarkStreams.identity()`.

Setup declares expected split counts for each supported benchmark; the table below checks the selected file. Fungal expression has a numerical target. E. coli uses class IDs 0, 1, and 2; the reader checks their integer values and reports their split distribution. The numeric codes alone do not identify the assay thresholds or establish which code means low, medium, or high expression.

For fungal expression, the local raw file has 7,089 rows, whereas the paper reports 7,056. Its retained count of 3,138 agrees with the paper but does not prove identical examples. For both benchmarks, the paper describes truncation while this implementation filters out long rows. Target assay provenance, units, transformations, and class thresholds require source validation. [Step 1's dataset audit](../stage3/01_blf_datasets.ipynb) holds the detailed evidence.

In [ ]:
spec = datasets.DATASETS[BENCHMARK]
rows = datasets.blf_benchmark_rows(BENCHMARK)
split_counts = rows[spec.split_column].value_counts().to_dict()
if spec.task != TASK or split_counts != EXPECTED_SPLITS:
    raise ValueError(f"Review the benchmark task or split changes before comparing: {split_counts}")
labels = rows[spec.label_column].to_numpy()
if not np.issubdtype(labels.dtype, np.number) or not np.isfinite(labels).all():
    raise ValueError("The retained target column must contain finite numeric values.")
if TASK == "classification":
    if not np.equal(labels, np.floor(labels)).all() or set(labels) != {0, 1, 2}:
        raise ValueError("E. coli requires the integer class IDs 0, 1, and 2.")
    display(pd.crosstab(rows[spec.split_column], rows[spec.label_column]).reindex(EXPECTED_SPLITS))

row_columns = ["sequence", "length", spec.label_column, spec.split_column]
row_texts = (
    json.dumps(row, ensure_ascii=False, separators=(",", ":"), default=lambda value: value.item())
    for row in rows[row_columns].itertuples(index=False, name=None)
)
rows_sha256 = input_fingerprint(row_texts)
# GridDataset preserves test-row order and casts regression targets to float32, classes to int64.
test_rows = rows.loc[rows[spec.split_column] == "test"]
expected_test_truth = test_rows[spec.label_column].to_numpy(
    dtype=np.int64 if TASK == "classification" else np.float32
)
display(pd.Series(split_counts, name="retained rows").reindex(EXPECTED_SPLITS).to_frame())

### Inspect the recorded inputs, and verify cache headers when available

A result records the source files, inputs, cache fingerprints, and environment of its producer. The reader can use those records after the code changes: the source currently reading a result is not the source that produced it. The explicitly selected reference above identifies the producer cohort used for comparison. Its source snapshot and verification history belong in the [run records](../../docs/run-records.md).

Each cache header identifies its encoder and ordered effective inputs. When headers are available under `CACHE_ROOT`, the next cell verifies those fields and records each header's hash. Completed records must match every available verified header. Missing or invalid headers remain visible below and block the final comparison; they do not prevent inspection of the recorded evidence. No cache is copied or created, and no embedding shard is opened.

Recorded cache and source fingerprints must be complete and consistent within the chosen cohort. This checks the records; it does not freshly verify large shard bytes, reproduce numerical results, or prove which modules a long-lived kernel had loaded. The environment reading these results may differ from the recorded producing environment.

In [ ]:
# Missing cache headers are a pending verification, not permission to invent their contents.
cache_headers = {}
cache_header_checks = []
for modality, encoder_name in blf.ENCODER_OF_MODALITY.items():
    metadata_path = CACHE_ROOT / BENCHMARK / encoder_name / "meta.json"
    if not metadata_path.is_file():
        cache_header_checks.append({
            "modality": modality, "status": "pending", "detail": "Cache header is unavailable.",
        })
        continue
    try:
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        expected_input = input_fingerprint(inputs_for_rows(ENCODERS[encoder_name], rows["sequence"]))
        if (metadata.get("encoder") != encoder_name
                or metadata.get("n_sequences") != len(rows)
                or metadata.get("input_sha256") != expected_input):
            raise ValueError("Header does not identify the current encoder inputs.")
        shard_count = metadata.get("n_shards")
        if type(shard_count) is not int or shard_count <= 0:
            raise ValueError("Expected a positive shard count.")
        cache_headers[modality] = {
            "meta_sha256": file_fingerprint(metadata_path),
            "filenames": {"meta.json", *(f"shard_{index:05d}.npz" for index in range(shard_count))},
        }
        cache_header_checks.append({
            "modality": modality, "status": "verified", "detail": "Header matches ordered encoder inputs.",
        })
    except (OSError, ValueError, TypeError, AttributeError) as error:
        cache_header_checks.append({"modality": modality, "status": "invalid", "detail": str(error)})

CACHE_HEADERS_VERIFIED = (
    set(cache_headers) == set(EXPECTED_WIDTHS)
    and all(check["status"] == "verified" for check in cache_header_checks)
)
display(pd.DataFrame(cache_header_checks).set_index("modality"))
print("Cache headers checked." if CACHE_HEADERS_VERIFIED
      else "Independent cache-header verification is pending or failed; final comparison stays blocked.")

# Require the producer's complete source list, without replacing it with this reader's hashes.
source_names = (
    "train.py", "blf.py", "codon_grid.py", "heads.py", "textcnn.py",
    "token_embeddings.py", "../datasets.py", "../sequences.py", "../encoders.py",
)

---

## Inspect completed runs for the declared comparison

Only final `.json` result files in `RESULT_DIRECTORY` are read. Checkpoints and `.partial` files are not completed results; an absent JSON means **not available in this snapshot**, not a failed fit. No files are copied or rewritten. Rerun this section after more results finish to refresh the snapshot.

A represented candidate must match the declared variant, seed, entropy weight, full training configuration, current ordered rows, encoder widths, and output size. Its complete recorded cache, source, and environment identity must also match the explicit historical reference. The filename locates that reference; file contents determine compatibility. Legacy records without identity are excluded. Equation-convolution runs remain separate from this appendix comparison, even when they are in the same directory. A record excluded from this analysis may still document completed work; exclusion does not establish that the experiment needs repeating.

The checks below separate record compatibility from independent artifact verification. Without cache headers, a compatible record can represent a planned fit, but it cannot make the final comparison ready. Each rejected file gets a reason so a malformed or unrelated record does not hide the rest of the inventory. The reference itself must pass the same checks.

In [ ]:
def check_protocol(result):
    """Require the declared experiment and complete recorded identity; return its grid key."""
    config = result.config
    key = (config["seed"], result.variant, config["entropy_lambda"])
    if type(config["seed"]) is not int or key not in planned_keys:
        raise ValueError("Variant, seed, or entropy weight is outside the planned grid.")
    expected_config = asdict(replace(BASE_CONFIG, seed=key[0], entropy_lambda=key[2]))
    if result.benchmark != BENCHMARK or config != expected_config:
        raise ValueError("Benchmark or full training configuration differs from the protocol.")
    identity = result.identity
    if not isinstance(identity, dict) or type(identity.get("version")) is not int or identity["version"] != 1:
        raise ValueError("Missing or unsupported run identity; legacy results are excluded.")
    json.dumps(identity, allow_nan=False)
    if identity.get("variant") != result.variant or identity.get("config") != config:
        raise ValueError("Result and embedded identity disagree.")
    if (identity.get("widths") != EXPECTED_WIDTHS or identity.get("n_outputs") != EXPECTED_OUTPUTS
            or identity.get("model_kwargs") != {}):
        raise ValueError("Model dimensions or arguments differ.")

    # These hashes describe the historical producer, not the currently imported reader.
    code_hashes = identity.get("code_sha256")
    if not isinstance(code_hashes, dict) or set(code_hashes) != set(source_names):
        raise ValueError("The recorded producer source list is incomplete or unexpected.")
    inputs = identity["inputs"]
    expected_inputs = {
        "benchmark": BENCHMARK, "task": TASK, "row_columns": row_columns,
        "rows_sha256": rows_sha256, "input_identity_status": "recorded_inputs",
    }
    if any(inputs.get(name) != value for name, value in expected_inputs.items()):
        raise ValueError("Ordered rows or input provenance differ from the current benchmark.")
    cache_files = inputs["cache_files"]
    if not isinstance(cache_files, dict) or set(cache_files) != set(EXPECTED_WIDTHS):
        raise ValueError("The recorded cache modalities are incomplete or unexpected.")
    fingerprint_groups = {"source": code_hashes}
    for modality, hashes in cache_files.items():
        if not isinstance(hashes, dict) or len(hashes) < 2:
            raise ValueError(f"{modality}: missing recorded cache fingerprints.")
        expected_files = {"meta.json", *(f"shard_{index:05d}.npz" for index in range(len(hashes) - 1))}
        if set(hashes) != expected_files:
            raise ValueError(f"{modality}: cache filenames must include metadata and contiguous shards.")
        if modality in cache_headers:
            header = cache_headers[modality]
            if set(hashes) != header["filenames"] or hashes["meta.json"] != header["meta_sha256"]:
                raise ValueError(f"{modality}: recorded cache files or header hash differ.")
        fingerprint_groups[modality] = hashes
    for name, hashes in fingerprint_groups.items():
        if any(not isinstance(value, str) or len(value) != 64
               or any(character not in "0123456789abcdef" for character in value)
               for value in hashes.values()):
            raise ValueError(f"{name}: missing or malformed SHA256 fingerprints.")

    environment = identity.get("environment")
    required_environment = {
        "python", "platform", "torch", "numpy", "scipy", "device", "threads", "interop_threads",
        "float32_matmul_precision", "deterministic_algorithms", "cudnn_deterministic",
        "cudnn_benchmark", "cudnn_allow_tf32", "cuda_matmul_allow_tf32", "cublas_workspace_config",
    }
    if isinstance(environment, dict) and str(environment.get("device")).startswith("cuda"):
        required_environment.update(("cuda", "device_name", "device_index", "cuda_device_count"))
    if not isinstance(environment, dict) or not required_environment <= environment.keys():
        raise ValueError("The producing environment record is incomplete.")
    if any(not isinstance(environment[name], str) or not environment[name]
           for name in ("python", "platform", "torch", "numpy", "scipy", "device")):
        raise ValueError("The producing environment has invalid version or device fields.")
    if any(type(environment[name]) is not int or environment[name] <= 0
           for name in ("threads", "interop_threads")):
        raise ValueError("The producing environment has invalid thread counts.")
    return key


def check_reference_cohort(result, reference_identity):
    """Keep the recorded producer, inputs, and environment together across candidates."""
    for field in COHORT_FIELDS:
        if result.identity[field] != reference_identity[field]:
            raise ValueError(f"Recorded {field} differs from the explicitly selected reference cohort.")

### Check completion and the selected epoch

`Result.load()` deserializes a file; it does not validate its contents. A usable result needs finite validation and test scores in the metric's range: −1 to 1 for Spearman, or 0 to 1 for accuracy. Accuracy is stored as a proportion; only displayed scores and differences are multiplied by 100 to give percentages and percentage points.

The epoch history must be contiguous, and the saved validation score must match the first maximum in that history. Undefined early validation scores are allowed; a run with no defined validation score cannot select a model. The stored `best_epoch` starts at zero; the displayed selection table adds one so that “epoch 1” means the first training epoch.

In [ ]:
def check_scores(result):
    """Reject incomplete or inconsistent scores before any validation selection."""
    for score in (result.val_score, result.test_score):
        if type(score) not in (int, float) or not np.isfinite(score) or not SCORE_MINIMUM <= score <= 1:
            raise ValueError("Validation and test scores must be finite and inside the declared metric range.")
    if type(result.epochs) is not int or not 1 <= result.epochs <= BASE_CONFIG.max_epochs:
        raise ValueError("Invalid completed epoch count.")
    if (len(result.history) != result.epochs
            or [entry["epoch"] for entry in result.history] != list(range(result.epochs))
            or type(result.best_epoch) is not int or not 0 <= result.best_epoch < result.epochs):
        raise ValueError("Epoch count, history, or selected epoch is inconsistent.")
    validation_history = [entry["val_score"] for entry in result.history]
    for score in validation_history:
        if score is not None and (type(score) not in (int, float)
                                  or not np.isfinite(score) or not SCORE_MINIMUM <= score <= 1):
            raise ValueError("Invalid validation score in the epoch history.")
    comparable = [-np.inf if score is None else score for score in validation_history]
    if (result.best_epoch != int(np.argmax(comparable))
            or result.val_score != comparable[result.best_epoch]):
        raise ValueError("Saved validation selection disagrees with the first best epoch.")

### Validate saved predictions before using them

Field presence is an inventory, not evidence that arrays are usable. The shared check requires one finite target and prediction for each current test row, the stored target order expected from the standard test loader, valid class IDs for classification, and a recomputed score agreeing with the saved score within `1e-12`. Regression targets use the training loader's float32 representation; classification targets use int64.

These checks operate alongside the recorded full-row and producer identities. Equal class labels alone cannot identify rows when many examples share a label. A record without predictions can remain aggregate evidence, but it cannot support prediction-level uncertainty. Present but invalid arrays exclude the record from comparison, because they contradict its claimed prediction evidence; the field-presence inventory still shows that record. Its absence never authorizes overwriting or automatically retraining it.

In [ ]:
def check_predictions(result):
    """Check stored arrays against the identified test rows, using the runner's same rule."""
    phasea_runner.validate_prediction_arrays(result, TASK, expected_test_truth)

In [ ]:
# Snapshot final files once; another process may publish additional results after this cell.
result_paths = sorted(RESULT_DIRECTORY.glob("*.json"))
inspected = []
checked = []
excluded = []
results_by_path = {}
reference_identity = None
try:
    reference_result = train.Result.load(REFERENCE_RESULT_PATH)
    check_protocol(reference_result)
    check_scores(reference_result)
    reference_identity = reference_result.identity
except (OSError, ValueError, TypeError, KeyError, AttributeError) as error:
    print(f"Reference is pending or invalid: {REFERENCE_RESULT_PATH.name}: {error}")

for path in result_paths:
    try:
        digest_before = file_fingerprint(path)
        result = train.Result.load(path)
        seed, variant, entropy_weight = check_protocol(result)
        check_scores(result)
        prediction_status, prediction_detail = "verified", "Ordered arrays and saved score agree."
        try:
            check_predictions(result)
        except (ValueError, TypeError, KeyError, AttributeError) as error:
            prediction_status = ("missing" if result.test_truth is None or result.test_prediction is None
                                 else "invalid")
            prediction_detail = str(error)
        if file_fingerprint(path) != digest_before:
            raise ValueError("The result changed while being read; refresh the snapshot.")
        record = {
            "seed": seed, "variant": variant, "entropy_lambda": entropy_weight,
            "val_score": result.val_score, "test_score": result.test_score,
            "best_epoch": result.best_epoch + 1, "path": path.relative_to(REPO_ROOT).as_posix(),
            "identity": result.identity, "file_sha256": digest_before,
            "has_test_truth": result.test_truth is not None,
            "has_test_prediction": result.test_prediction is not None,
            "prediction_status": prediction_status, "prediction_detail": prediction_detail,
        }
        checked.append(record)
        if prediction_status == "invalid":
            raise ValueError(f"Invalid saved prediction arrays: {prediction_detail}")
        if reference_identity is None:
            raise ValueError("The explicit reference is unavailable or invalid; cohort verification is blocked.")
        check_reference_cohort(result, reference_identity)
        inspected.append(record)
        results_by_path[record["path"]] = result
    except (OSError, ValueError, TypeError, KeyError, AttributeError) as error:
        excluded.append({"file": path.name, "reason": str(error)})

result_columns = [
    *KEY_COLUMNS, "val_score", "test_score", "best_epoch", "path", "identity", "file_sha256",
    "has_test_truth", "has_test_prediction", "prediction_status", "prediction_detail",
]
# checked_results passed individual protocol/history checks; results also match the reference cohort.
checked_results = pd.DataFrame(checked, columns=result_columns)
results = pd.DataFrame(inspected, columns=result_columns)
print(f"{len(result_paths)} completed files inspected; {len(results)} match the protocol and recorded reference cohort.")
print("Reference:", REFERENCE_RESULT_PATH.relative_to(REPO_ROOT).as_posix())
if excluded:
    display(pd.DataFrame(excluded))

# Presence remains separate from the stricter array validation reported beside it.
prediction_availability = checked_results.groupby(
    ["has_test_truth", "has_test_prediction"]
).size().rename("matching_records").reset_index()
print("Saved per-row fields among protocol/history-checked records (presence only):")
display(prediction_availability.rename(columns={
    "has_test_truth": "test targets stored", "has_test_prediction": "test predictions stored",
    "matching_records": "records",
}))
if not checked_results.empty:
    display(checked_results[[*KEY_COLUMNS, "prediction_status", "prediction_detail"]])

### Distinguish represented fits from a ready comparison

Every candidate in this declared historical comparison needs exactly one compatible record. This requirement does not prescribe the candidate or repetition schedule for another experiment. Duplicate records block comparison; neither file recency nor test performance resolves a duplicate. Runs must agree with the selected reference on recorded cache, source, and producing-environment identities. A different source or environment requires an explicit separate comparison, even when each record passes its individual checks. A future run produced by changed code does not automatically fill a gap in this historical cohort.

The table describes the inspected snapshot without selecting a winner from an incomplete lambda search. Each appendix fit represents only its own seed and candidate in the declared plan. Equation fits remain outside that plan. Coverage is computed from eligible records; `not_represented` means no compatible record qualified for that slot. A file may still be present but excluded. The saved-work inventory below distinguishes absence from incompatibility; neither status measures the team's progress or calls for another fit.

**Represented** means one record passes the protocol, history, and reference-cohort checks. It does not mean the cache artifacts were independently verified. Every declared fit, one recorded cohort, and verified headers for every required modality are needed before selection, aggregate statistics, or figures. Missing headers and excluded records stay visible; nothing is retrained to repair a gap.

In [ ]:
counts = results.groupby(KEY_COLUMNS).size().rename("matching_files").reset_index()
coverage = plan.merge(counts, on=KEY_COLUMNS, how="left")
coverage["matching_files"] = coverage["matching_files"].fillna(0).astype(int)
coverage["status"] = coverage["matching_files"].map(
    lambda count: "not_represented" if count == 0 else "represented" if count == 1 else "duplicate"
)
display(coverage.pivot(index="candidate", columns="seed", values="status").reindex(
    [label for label, _, _ in CANDIDATES]
))

# Compare recorded identities, not this reader's code or current kernel environment.
cohort_signatures = results["identity"].map(
    lambda identity: json.dumps({field: identity[field] for field in COHORT_FIELDS}, sort_keys=True, allow_nan=False)
)
one_cohort = cohort_signatures.nunique() == 1
if cohort_signatures.nunique() > 1:
    print("Comparison blocked: more than one recorded input/code/environment cohort.")

GRID_COMPLETE = coverage["matching_files"].eq(1).all() and one_cohort
COMPARISON_READY = GRID_COMPLETE and CACHE_HEADERS_VERIFIED
print(f"{coverage['matching_files'].eq(1).sum()}/{len(plan)} planned candidates are represented exactly once.")
print("Independent cache-header verification:", "verified" if CACHE_HEADERS_VERIFIED else "pending or failed")
print("Ready for the declared comparison." if COMPARISON_READY
      else "Comparison pending: resolve missing, duplicate, incompatible, or unverified inputs.")

### Inspect saved work without scheduling new fits

The table below distinguishes local evidence, missing prediction fields, incompatibility with this comparison, and checkpoints requiring review. It never opens token caches or calls training. A missing reference blocks comparison, not inspection; a saved result awaiting its reference remains visible. `not_found_locally` means that no result or checkpoint was found at the inspected location. A teammate may already have completed or be running that experiment elsewhere.

Prediction fields can be absent or invalid, and a completed record can belong to a different protocol. Preserve those records and review their meaning; none of these statuses authorizes a refit. Reported work, an inspected artifact, and eligibility for this comparison are different evidence states. The retained runner's exact execution checks concern cache bytes, current source, device, and checkpoint identity; they are separate from this historical inspection and remain uninvoked here.

In [ ]:
# Inspect local evidence only; retain the table name for existing consumers.
preview_rows = []
for item in plan.itertuples(index=False):
    config = replace(BASE_CONFIG, seed=item.seed, entropy_lambda=item.entropy_lambda)
    expected_path = phasea_runner.result_path(RESULT_DIRECTORY, BENCHMARK, item.variant, config)
    key_mask = (checked_results["seed"].eq(item.seed)
                & checked_results["variant"].eq(item.variant)
                & checked_results["entropy_lambda"].eq(item.entropy_lambda))
    stored = checked_results.loc[key_mask]
    matched = results.loc[
        results["seed"].eq(item.seed) & results["variant"].eq(item.variant)
        & results["entropy_lambda"].eq(item.entropy_lambda)
    ]
    if len(stored) > 1:
        status, detail = "duplicate_requires_review", "More than one protocol-compatible record."
    elif len(stored) == 1 and stored.iloc[0]["prediction_status"] != "verified":
        status = ("predictions_absent" if stored.iloc[0]["prediction_status"] == "missing"
                  else "predictions_invalid")
        detail = stored.iloc[0]["prediction_detail"]
    elif len(stored) == 1 and reference_identity is None:
        status, detail = "reference_pending", "A saved fit exists; the declared reference must be reviewed."
    elif len(matched) == 1:
        status, detail = "recorded_evidence_available", "Analysis evidence; execution reuse needs the runner's identity check."
    elif expected_path in result_paths or expected_path.is_file() or len(stored):
        status, detail = "incompatible_with_comparison", "A completed record is preserved; review why it differs."
    elif expected_path.with_suffix(".checkpoint").is_file():
        status, detail = "checkpoint_requires_review", "The runner must verify resumability; this reader does not load checkpoints."
    else:
        status, detail = "not_found_locally", "No result or checkpoint found here; work elsewhere may already exist."
    preview_rows.append({
        "seed": item.seed, "candidate": item.candidate, "variant": item.variant,
        "entropy_lambda": item.entropy_lambda, "status": status, "detail": detail,
        "expected_path": expected_path.relative_to(REPO_ROOT).as_posix(),
    })
execution_preview = pd.DataFrame(preview_rows)
display(execution_preview[["seed", "candidate", "status", "detail"]])
print("Saved-work inspection only; these statuses do not schedule training or establish work elsewhere.")

---

## Select on validation, then compare test scores

Within each seed, choose the positive MIL entropy weight with the highest validation score. Separately choose the highest-validation single encoder from the explicit `BASELINE_VARIANTS` list as the comparator. Exact ties follow the order declared in Setup. MIL with zero entropy remains a separate ablation and never participates in the positive-lambda selection.

For fusion method $m$ and seed $s$, the paired difference is

$$
\Delta_{m,s} = S^{\mathrm{test}}_{m,s} - S^{\mathrm{test}}_{b_s,s},
\qquad
b_s = \underset{b\in\{\mathrm{DNA,RNA,Protein}\}}{\operatorname{argmax}}\;S^{\mathrm{val}}_{b,s}.
$$

Here $S$ is Spearman correlation for regression or accuracy as a proportion for classification. Spearman compares the ranking of predicted expression with the ranking of supplied targets; accuracy is the fraction of correctly predicted classes. A positive $\Delta$ means the fusion procedure outscored the **validation-selected single encoder** for that seed. Classification differences are multiplied by 100 only for display, giving percentage points rather than percent relative improvement.

The baseline may differ between seeds; this evaluates a repeated selection procedure, not a comparison with one permanently fixed encoder. The shared training loop already calculates test scores for every candidate. The code below does not use them to choose the lambda or baseline. A supplied benchmark split inspected during development supports a reproduction comparison, not an untouched confirmatory evaluation.

In [ ]:
# Reset derived tables on every run so an incomplete refresh cannot retain an old comparison.
selected = pd.DataFrame()
choices = pd.DataFrame()
paired = pd.DataFrame()
MAIN_METHODS = ["DNA", "RNA", "Protein", "Concat", "MIL", "Cross"]
if COMPARISON_READY:
    ordered_results = plan.merge(results, on=KEY_COLUMNS, how="left", validate="one_to_one")
    selected_rows, choice_rows, difference_rows = [], [], []
    for seed in SEEDS:
        seed_results = ordered_results[ordered_results["seed"] == seed]
        single_runs = seed_results[seed_results["variant"].isin(BASELINE_VARIANTS)]
        positive_mil = seed_results[
            seed_results["variant"].eq("mil") & seed_results["entropy_lambda"].isin(POSITIVE_LAMBDAS)
        ]
        baseline = single_runs.loc[single_runs["val_score"].idxmax()]
        chosen_mil = positive_mil.loc[positive_mil["val_score"].idxmax()]
        choice_rows.append({
            "seed": seed, "single encoder": baseline["candidate"],
            "single validation": baseline["val_score"], "MIL lambda": chosen_mil["entropy_lambda"],
            "MIL validation": chosen_mil["val_score"],
        })
        # Fixed candidates plus the one validation-selected positive-lambda MIL run.
        reported_runs = pd.concat([seed_results[seed_results["entropy_lambda"].eq(0)], chosen_mil.to_frame().T])
        for _, run in reported_runs.iterrows():
            method = "MIL" if run["variant"] == "mil" and run["entropy_lambda"] > 0 else run["candidate"]
            selected_rows.append({"seed": seed, "method": method, "val_score": run["val_score"],
                                  "test_score": run["test_score"], "best_epoch": run["best_epoch"], "path": run["path"]})
            if method in ("Concat", "MIL", "Cross", "MIL without entropy"):
                difference_rows.append({"seed": seed, "method": method,
                                        "delta": run["test_score"] - baseline["test_score"]})
    selected = pd.DataFrame(selected_rows)
    choices = pd.DataFrame(choice_rows)
    paired = pd.DataFrame(difference_rows)
    choice_display = choices.set_index("seed").copy()
    choice_display[["single validation", "MIL validation"]] *= DISPLAY_SCALE
    print("Validation-selection display units:", benchmark_settings["score_label"].replace("Test ", ""))
    display(choice_display.round(4))
else:
    print("No model selection: the full planned grid is not ready.")

### Read the scores and their variation

The first table retains each seed, its validation and test score, and its selected epoch. The summary reports the arithmetic mean and sample standard deviation (`ddof=1`) across the declared training seeds. These describe variation across training seeds on this fixed split, including per-seed validation selection where it is part of the reported procedure; they are not confidence intervals, biological replicates, or evidence of a significant gain.

Spearman scores stay on their correlation scale. Accuracy tables show percentages, and their differences and standard deviations are in percentage points. The calculation tables keep accuracy as proportions internally; display conversion must not change model selection or saved records.

The separate ablation table compares MIL without entropy with the same validation-selected single-encoder baseline. Do not silently replace the positive-lambda MIL procedure with this ablation if its test score is larger.

In [ ]:
if COMPARISON_READY:
    score_display = selected.drop(columns="path").sort_values(["method", "seed"]).copy()
    score_display[["val_score", "test_score"]] *= DISPLAY_SCALE
    print("Score display units:", benchmark_settings["score_label"].replace("Test ", ""))
    display(score_display.round(4))
    summary = selected.groupby("method")["test_score"].agg(["count", "mean", "std"])
    summary_display = summary.reindex(MAIN_METHODS).rename(columns={"count": "seeds", "std": "seed SD"}).copy()
    summary_display[["mean", "seed SD"]] *= DISPLAY_SCALE
    display(summary_display.round(4))
    ablation = selected[selected["method"] == "MIL without entropy"].merge(
        paired, on=["seed", "method"], validate="one_to_one"
    )
    ablation_display = ablation[["seed", "val_score", "test_score", "delta"]].copy()
    ablation_display[["val_score", "test_score", "delta"]] *= DISPLAY_SCALE
    display(ablation_display.round(4))
else:
    print("Score summaries wait for a complete, comparable grid.")

### Test performance and paired gains

The first figure shows each training seed and the mean for every main method. The second subtracts each seed's validation-selected single-encoder score from its fusion score. The zero line marks equal performance. Points use the same seed colors across both figures; show the observations directly rather than implying a distribution with a boxplot or confidence band from a few seeds.

Compare the pattern with [BioLangFusion's Table 1](../../Papers/core/BioLangFusion.pdf), keeping dataset and implementation differences in view. The paper reports E. coli accuracy in percent; its accuracy gaps are percentage-point differences. Numerical closeness to a paper value alone does not validate a reproduction.

These plots use aggregate scores. The inventory above distinguishes prediction-field presence from verified arrays. Aggregate scores alone cannot recover residuals, prediction scatterplots, or row-bootstrap intervals. The next section uses saved predictions only after their values, shapes, and ordered test targets pass the separate checks; it does not load model weights.

In [ ]:
if COMPARISON_READY:
    score_matrix = selected.pivot(index="method", columns="seed", values="test_score").reindex(MAIN_METHODS)
    score_matrix = score_matrix * DISPLAY_SCALE
    positions = np.arange(len(MAIN_METHODS))
    offsets = np.linspace(-0.15, 0.15, len(SEEDS))
    seed_colors = ("#0072B2", "#D55E00", "#009E73")
    figure, axis = plt.subplots(figsize=(9, 4.5), layout="constrained")
    for seed, offset, color in zip(SEEDS, offsets, seed_colors):
        axis.scatter(positions + offset, score_matrix[seed], label=f"Seed {seed}", color=color, s=48)
    axis.scatter(positions, score_matrix.mean(axis=1), marker="_", s=250, color="black", label="Mean", zorder=3)
    axis.set(xticks=positions, xticklabels=MAIN_METHODS, ylabel=benchmark_settings["score_label"],
             title=f"{benchmark_settings['label']}: {len(SEEDS)} training seeds on the supplied split")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    plt.show()
else:
    print("Performance figure pending the complete grid.")

In [ ]:
if COMPARISON_READY:
    fusion_methods = ["Concat", "MIL", "Cross"]
    delta_matrix = paired.pivot(index="method", columns="seed", values="delta").reindex(fusion_methods)
    delta_matrix = delta_matrix * DISPLAY_SCALE
    positions = np.arange(len(fusion_methods))
    figure, axis = plt.subplots(figsize=(8, 4.5), layout="constrained")
    for seed, offset, color in zip(SEEDS, offsets, seed_colors):
        axis.scatter(positions + offset, delta_matrix[seed], label=f"Seed {seed}", color=color, s=48)
    axis.scatter(positions, delta_matrix.mean(axis=1), marker="_", s=250, color="black", label="Mean", zorder=3)
    axis.axhline(0, color="gray", linestyle="--", linewidth=1)
    axis.set(xticks=positions, xticklabels=fusion_methods, ylabel=benchmark_settings["difference_label"],
             title="Fusion minus the validation-selected single encoder, within each seed")
    axis.grid(axis="y", alpha=0.25)
    axis.legend(ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.12))
    plt.show()
else:
    print("Paired-difference figure pending the complete grid.")

### Paired uncertainty from the same test rows

For a selected fusion model and its validation-selected baseline, resample test-row positions with replacement and use the same positions for both models. Compute their score difference in each resample, then report the 2.5th and 97.5th percentiles. The recorded bootstrap seed and draw count make this Monte Carlo calculation repeatable.

These are **per-training-seed intervals conditional on the fitted models and this test split**. They do not include uncertainty from training again, changing validation selection, sampling a different population, or dependence among biologically related sequences. The notebook never stacks the same test rows across training seeds as independent observations. The seed summary above remains a separate description of between-run variation, including any per-seed validation selection.

A pair is evaluated only when the full declared comparison is ready and both prediction arrays pass validation. Missing predictions, undefined resampled correlations, or inconsistent records produce an explicit unavailable result rather than an inferred interval or a replacement fit. The intervals are individual descriptive intervals without a multiple-comparison correction. A fraction of bootstrap differences above zero is a resampling summary, not a posterior probability that a biological hypothesis is true.

The figure places the observed paired difference beside its individual 95% interval for each method and seed, including the zero-entropy MIL ablation. Colors follow the earlier figures, and the legend identifies each seed's validation-selected baseline. The zero line marks equal performance; an interval crossing zero leaves both signs compatible with this conditional resampling summary.

In [ ]:
uncertainty_rows = []
if COMPARISON_READY:
    for seed in SEEDS:
        seed_results = selected[selected["seed"] == seed].set_index("method")
        baseline_name = choices.loc[choices["seed"] == seed, "single encoder"].iloc[0]
        baseline_path = seed_results.loc[baseline_name, "path"]
        baseline_result = results_by_path[baseline_path]
        for method in paired.loc[paired["seed"] == seed, "method"]:
            fusion_path = seed_results.loc[method, "path"]
            fusion_result = results_by_path[fusion_path]
            row = {"seed": seed, "method": method, "baseline": baseline_name,
                   "fusion_path": fusion_path, "baseline_path": baseline_path,
                   "status": "unavailable", "detail": ""}
            try:
                check_predictions(fusion_result)
                check_predictions(baseline_result)
                interval = train.paired_bootstrap(
                    fusion_result, baseline_result, TASK,
                    draws=BOOTSTRAP_DRAWS, seed=BOOTSTRAP_SEED,
                )
                row.update(interval, status="available")
            except (ValueError, TypeError, KeyError, AttributeError) as error:
                row["detail"] = str(error)
            uncertainty_rows.append(row)
uncertainty = pd.DataFrame(uncertainty_rows)
UNCERTAINTY_READY = (
    COMPARISON_READY and len(uncertainty) == len(paired) and len(uncertainty) > 0
    and uncertainty["status"].eq("available").all()
)
if not uncertainty.empty:
    uncertainty_display = uncertainty.drop(columns=["fusion_path", "baseline_path"]).copy()
    for name in ("difference", "ci_low", "ci_high"):
        if name in uncertainty_display:
            uncertainty_display[name] *= DISPLAY_SCALE
    print(benchmark_settings["difference_label"], f"| {BOOTSTRAP_DRAWS} draws, bootstrap seed {BOOTSTRAP_SEED}")
    display(uncertainty_display.round(4))
else:
    print("Paired uncertainty waits for a complete, independently checked comparison.")

In [ ]:
if UNCERTAINTY_READY:
    interval_methods = ["Concat", "MIL", "Cross", "MIL without entropy"]
    positions = np.arange(len(interval_methods))
    interval_figure, interval_axis = plt.subplots(figsize=(9.5, 5.2), layout="constrained")
    for seed, offset, color in zip(SEEDS, offsets, seed_colors):
        seed_intervals = uncertainty[uncertainty["seed"] == seed].set_index("method").reindex(interval_methods)
        baseline_name = seed_intervals["baseline"].iloc[0]
        # Draw percentile endpoints directly; the observed difference need not be inside them.
        interval_axis.hlines(
            positions + offset, seed_intervals["ci_low"] * DISPLAY_SCALE,
            seed_intervals["ci_high"] * DISPLAY_SCALE, color=color, linewidth=2,
        )
        interval_axis.scatter(
            seed_intervals["difference"] * DISPLAY_SCALE, positions + offset,
            color=color, s=42, label=f"Seed {seed} vs {baseline_name}", zorder=3,
        )
    interval_axis.axvline(0, color="gray", linestyle="--", linewidth=1)
    interval_axis.set(
        yticks=positions, yticklabels=interval_methods, xlabel=benchmark_settings["difference_label"],
        title="Fusion minus the validation-selected single encoder\nIndividual conditional 95% paired intervals",
    )
    interval_axis.invert_yaxis()
    interval_axis.grid(axis="x", alpha=0.25)
    interval_axis.legend(ncol=len(SEEDS), loc="upper center", bbox_to_anchor=(0.5, -0.15))
    plt.show()
else:
    print("Paired-interval figure waits for verified predictions and available intervals for every pair.")

### Keep a small analysis record

`analysis_manifest` records all candidate and selected result paths and their SHA256 hashes, benchmark-row identity, declared protocol, validation choices, metric units, and bootstrap settings and outcomes. Keeping the losing validation candidates makes the selection auditable as well as the displayed winners. It describes an analysis of existing results, not a new training run or independent verification of every cache byte. Producer identities remain attached to their own result files; the reading notebook and its two analysis helpers are fingerprinted separately.

The manifest is prepared in memory only after the comparison is ready. This notebook does not automatically write or overwrite an artifact. Preserve the reviewed manifest with a separately authorized analysis export, and link it from the existing [run records](../../docs/run-records.md); no manual completion table needs to be refreshed.

In [ ]:
analysis_manifest = None
if COMPARISON_READY:
    result_details = results.set_index("path")
    selected_files = [
        {"path": row.path, "sha256": result_details.loc[row.path, "file_sha256"],
         "seed": int(row.seed), "method": row.method,
         "prediction_status": result_details.loc[row.path, "prediction_status"]}
        for row in selected.itertuples(index=False)
    ]
    # Replace unavailable numeric cells with JSON null without rounding available values.
    uncertainty_records = uncertainty.astype(object).where(pd.notna(uncertainty), None).to_dict(orient="records")
    analysis_manifest = {
        "format_version": 1, "kind": "BioLangFusion Phase A analysis of saved results",
        "benchmark": BENCHMARK, "task": TASK, "rows_sha256": rows_sha256,
        "splits": dict(EXPECTED_SPLITS), "metric": METRIC,
        "stored_score_units": "accuracy proportion" if TASK == "classification" else "Spearman correlation",
        "display_scale": DISPLAY_SCALE,
        "protocol": {"base_config": asdict(BASE_CONFIG), "seeds": list(SEEDS),
                     "positive_lambdas": list(POSITIVE_LAMBDAS),
                     "candidates": plan.to_dict(orient="records")},
        "reference_path": REFERENCE_RESULT_PATH.relative_to(REPO_ROOT).as_posix(),
        "recorded_producer_cohort": {key: reference_identity[key] for key in COHORT_FIELDS},
        "cache_headers_verified": bool(CACHE_HEADERS_VERIFIED),
        "cache_shard_bytes_reverified": False,
        "reader_notebook_sha256": file_fingerprint(CODE_DIR / "stage4" / "tier2_blf_phaseA.ipynb"),
        "reader_helper_sha256": {
            "Code/shared_code/fusion/train.py": file_fingerprint(CODE_DIR / "shared_code" / "fusion" / "train.py"),
            "Code/stage4/run_blf_phaseA.py": file_fingerprint(CODE_DIR / "stage4" / "run_blf_phaseA.py"),
        },
        "candidate_files": [
            {"path": row.path, "sha256": row.file_sha256, "seed": int(row.seed),
             "variant": row.variant, "entropy_lambda": float(row.entropy_lambda)}
            for row in results.itertuples(index=False)
        ],
        "selected_files": selected_files,
        "validation_choices": choices.to_dict(orient="records"),
        "bootstrap": {"draws": BOOTSTRAP_DRAWS, "seed": BOOTSTRAP_SEED,
                      "method": "paired test-row percentile intervals, separately per training seed",
                      "all_pairs_available": bool(UNCERTAINTY_READY),
                      "comparisons": uncertainty_records},
    }
    # Check that the proposed record can be exported as strict JSON; do not save a file.
    json.dumps(analysis_manifest, allow_nan=False)
    print(f"Analysis manifest prepared in memory for {len(selected_files)} selected result files; no file written.")
else:
    print("No analysis manifest: the declared comparison is not ready.")

---

## Interpret before extending the experiment

A complete grid makes the specified comparison readable; it does not settle faithful paper reproduction or biological validity. Review whether each fusion gain is consistent across the declared seeds, which positive MIL weight validation selects, and whether the zero-entropy ablation changes the interpretation. A fusion gain may reflect complementary predictive information, differences in trainable capacity, or optimization. A gain alone does not establish complementary information, an independent source of molecular evidence, or a biological mechanism.

Before stronger claims, resolve the retained-row correspondence and filtering-versus-truncation issue, trace the target assay and transformations, and review the convolution, channel-width, and unimodal-baseline choices. The NT cache has 512 channels; the paper's 4,107-feature description also needs reconciliation. Use the dataset audit for exact-sequence overlap in the selected benchmark; an absence of exact overlap does not establish independence from homologous sequences, shared composition, or encoder pretraining data.

Run lineage belongs in the completed result identities and [run records](../../docs/run-records.md), including producing code/commit, local differences, environment, and retained inputs. Source fingerprints alone cannot establish which module objects a long-lived kernel executed. When reporting this comparison, identify the exact result paths stored in `selected` and retain the validation choices. The in-memory analysis manifest retains those selections and file hashes; this notebook does not write a run record or export artifacts automatically.

A pending inventory calls for review of available and reported work, not automatic replacement fits. Reconcile the intended Equation 1 protocol, the codon-RNA control, and the repetition and uncertainty rules with the contributing implementation before specifying any additional experiment. Existing appendix evidence remains a separate analysis. The paired intervals here concern fixed fitted models and test rows. Further training-seed uncertainty, encoder swaps, new fusion designs, and additional benchmarks need separate decisions after this comparison is reviewed. See the [Stage 4 plan](README.md) and [Gantt workbook](../../docs/Research_Project_Plan_Gantt.xlsx), `Details` items 17, 19, and 24.